# z611 - Mi Serie (global) sobre grano Cliente-Producto
Ratios de grupo GLOBALES (cruzan todos los clientes, como se definio) sobre `tb_features_CP601.parquet`.

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'CP602',
    'features_path': '/home/ds/exp/CP601/tb_features_CP601.parquet',
    'productos_path': '/home/ds/datasets/tb_productos.txt',
    'grupos': [['cat1'], ['cat2'], ['cat3'], ['brand']],
    'metricas': ['tn', 'tn_media_12']
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/CP602


In [4]:
df = pl.read_parquet(PARAM['features_path'])

tb_productos = pl.read_csv(PARAM['productos_path'], separator="\t").select(
    ["product_id", "cat1", "cat2", "cat3", "brand", "sku_size", "descripcion"]
)

df = df.join(tb_productos, on="product_id", how="left")
print(df.shape)

(16648066, 61)


## Ratios leave-one-out GLOBALES
Grupo = todas las filas (de todos los clientes) con esa categoria en ese periodo, excluyendo la fila propia (este par cliente-producto puntual).

In [5]:
def ratio_leave_one_out(df, group_cols, metric):
    nombre_grupo = "_".join(group_cols)
    agg = df.group_by(group_cols + ["periodo"]).agg(
        pl.col(metric).sum().alias("_suma_grupo"),
        pl.len().alias("_n_grupo")
    )
    out = df.join(agg, on=group_cols + ["periodo"], how="left")
    out = out.with_columns(
        (
            (pl.col("_suma_grupo") - pl.col(metric))
            / (pl.col("_n_grupo") - 1).clip(lower_bound=1)
        ).alias(f"{metric}_prom_{nombre_grupo}_excl")
    )
    out = out.with_columns(
        (pl.col(metric) / (pl.col(f"{metric}_prom_{nombre_grupo}_excl") + 1e-6)).alias(
            f"ratio_{metric}_{nombre_grupo}"
        )
    )
    return out.drop(["_suma_grupo", "_n_grupo", f"{metric}_prom_{nombre_grupo}_excl"])

for grupo in PARAM['grupos']:
    for metrica in PARAM['metricas']:
        df = ratio_leave_one_out(df, grupo, metrica)

for metrica in PARAM['metricas']:
    df = ratio_leave_one_out(df, ["descripcion"], metrica)

## Tamano vecino
Vecino de tamano por `descripcion` (mismo que z604). El self-join ahora incluye `customer_id`: se trae la venta de ESE MISMO cliente en la presentacion vecina.

In [6]:
vecinos = tb_productos.sort(["descripcion", "sku_size"]).with_columns([
    pl.col("product_id").shift(1).over("descripcion").alias("product_id_tam_ant"),
    pl.col("product_id").shift(-1).over("descripcion").alias("product_id_tam_post"),
]).select(["product_id", "product_id_tam_ant", "product_id_tam_post"])

df = df.join(vecinos, on="product_id", how="left")

In [7]:
tn_por_cliente_producto_periodo = df.select(["customer_id", "product_id", "periodo", "tn"])

df = df.join(
    tn_por_cliente_producto_periodo.rename({"product_id": "product_id_tam_ant", "tn": "tn_tam_ant"}),
    on=["customer_id", "product_id_tam_ant", "periodo"],
    how="left"
)
df = df.join(
    tn_por_cliente_producto_periodo.rename({"product_id": "product_id_tam_post", "tn": "tn_tam_post"}),
    on=["customer_id", "product_id_tam_post", "periodo"],
    how="left"
)

df = df.drop(["product_id_tam_ant", "product_id_tam_post"])

## Guardar

In [8]:
salida = os.path.join(ruta, "tb_features_CP602.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/CP602/tb_features_CP602.parquet
(16648066, 73)
